# DensePose Generator for PromptDresser Custom Images

This notebook generates DensePose visualizations for your custom images using Google Colab's free GPU.

**Instructions:**
1. Run cells in order from top to bottom
2. Upload your custom person images (supports multiple images)
3. Download the generated DensePose files
4. Copy them to your local `DATA/custom/test/image-densepose/` folder

**Runtime:** GPU recommended (Runtime → Change runtime type → GPU)

## Step 1: Install Detectron2 and DensePose

This takes about 2-3 minutes. You'll see compilation messages - this is normal.

In [ ]:
import sys
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install Detectron2
print("\n" + "="*60)
print("Installing Detectron2...")
print("="*60)
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

# Install DensePose
print("\n" + "="*60)
print("Installing DensePose...")
print("="*60)
!git clone https://github.com/facebookresearch/detectron2
!pip install -e detectron2/projects/DensePose

print("\n✓ Installation complete!")

## Step 2: Import Libraries and Setup

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files
import zipfile
import os

# Detectron2 imports
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

# DensePose imports
sys.path.insert(0, 'detectron2/projects/DensePose')
from densepose import add_densepose_config
from densepose.vis.densepose_results import DensePoseResultsFineSegmentationVisualizer
from densepose.vis.extractor import DensePoseResultExtractor

print("✓ All libraries imported successfully!")

## Step 3: Configure DensePose Model

In [ ]:
cfg = get_cfg()
add_densepose_config(cfg)

# Use R_50_FPN_s1x model (good balance of speed and accuracy)
cfg.merge_from_file("detectron2/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml")
cfg.MODEL.WEIGHTS = "https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl"
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading DensePose model...")
predictor = DefaultPredictor(cfg)
print("✓ Model loaded and ready!")
print(f"Using device: {cfg.MODEL.DEVICE.upper()}")

## Step 4: Upload Your Custom Images

Click "Choose Files" and upload all your person images (PNG format).
The notebook will automatically process all uploaded files.

In [ ]:
# Create directories
os.makedirs('input_images', exist_ok=True)
os.makedirs('output_densepose', exist_ok=True)

print("Please upload your custom person images (PNG format):")
print("\n" + "="*60)

uploaded = files.upload()

# Move uploaded files to input directory
for filename in uploaded.keys():
    with open(f'input_images/{filename}', 'wb') as f:
        f.write(uploaded[filename])
    print(f"✓ Uploaded: {filename}")

print("="*60)
print(f"Total files uploaded: {len(uploaded)}")

## Step 5: Generate DensePose Visualizations

This will process each image and create DensePose maps.

In [ ]:
def generate_densepose(image_path, predictor):
    """Generate DensePose visualization for an image"""
    # Read image
    image = cv2.imread(str(image_path))
    if image is None:
        print(f"    ✗ Failed to read: {image_path}")
        return None
    
    # Run prediction
    with torch.no_grad():
        outputs = predictor(image)["instances"]
    
    # Check if person detected
    if len(outputs) == 0:
        print(f"    ⚠️  No person detected in image")
        return np.zeros_like(image)
    
    # Extract DensePose results - this returns a list of DensePoseResult objects
    extractor = DensePoseResultExtractor()
    results = extractor(outputs)
    
    if len(results) == 0:
        print(f"    ⚠️  No DensePose result extracted")
        return np.zeros_like(image)
    
    # Get bounding boxes
    boxes_xyxy = outputs.pred_boxes.tensor
    
    # Convert XYXY to XYWH format
    x1 = boxes_xyxy[:, 0]
    y1 = boxes_xyxy[:, 1]
    x2 = boxes_xyxy[:, 2]
    y2 = boxes_xyxy[:, 3]
    w = x2 - x1
    h = y2 - y1
    boxes_xywh = torch.stack([x1, y1, w, h], dim=1)
    
    # Create black background
    black_background = np.zeros_like(image)
    
    # Process only first detected person
    # Visualizer expects tuple of (list_of_results, tensor_of_boxes)
    visualizer = DensePoseResultsFineSegmentationVisualizer()
    
    # Take only first person: results is already a list, boxes_xywh is a tensor
    # We need first element of results list and first row of boxes tensor
    first_result = [results[0]]  # Wrap single result in list
    first_box = boxes_xywh[0:1]  # Keep as 2D tensor (1 x 4)
    
    vis_image = visualizer.visualize(black_background, (first_result, first_box))
    
    return vis_image


# Process all uploaded images
input_dir = Path('input_images')
output_dir = Path('output_densepose')

image_files = sorted(input_dir.glob('*.png'))

print("="*60)
print("Generating DensePose visualizations...")
print("="*60 + "\n")

for i, img_path in enumerate(image_files, 1):
    print(f"[{i}/{len(image_files)}] Processing: {img_path.name}")
    
    # Generate DensePose
    vis_image = generate_densepose(img_path, predictor)
    
    if vis_image is not None:
        # Save as .jpg (matching VITON-HD format)
        output_name = img_path.stem + '.jpg'
        output_path = output_dir / output_name
        cv2.imwrite(str(output_path), vis_image)
        print(f"    ✓ Saved: {output_name}")
        print(f"    Size: {vis_image.shape[1]}x{vis_image.shape[0]}\n")
    else:
        print(f"    ✗ Failed to process\n")

print("="*60)
print("✓ DensePose generation complete!")
print("="*60)

## Step 6: Preview Generated DensePose Images

In [ ]:
# Display original and DensePose side-by-side
output_files = sorted(Path('output_densepose').glob('*.jpg'))

fig, axes = plt.subplots(len(output_files), 2, figsize=(10, 5*len(output_files)))

if len(output_files) == 1:
    axes = axes.reshape(1, -1)

for i, output_file in enumerate(output_files):
    # Original image
    img_name = output_file.stem + '.png'
    orig_path = Path('input_images') / img_name
    orig_img = cv2.imread(str(orig_path))
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    
    # DensePose image
    dense_img = cv2.imread(str(output_file))
    dense_img = cv2.cvtColor(dense_img, cv2.COLOR_BGR2RGB)
    
    # Display
    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title(f'Original: {img_name}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(dense_img)
    axes[i, 1].set_title(f'DensePose: {output_file.name}')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ Showing {len(output_files)} DensePose visualizations")

## Step 7: Download Generated DensePose Files

This will create a zip file with all DensePose images and download it.

In [ ]:
# Create zip file
zip_filename = 'densepose_results.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in Path('output_densepose').glob('*.jpg'):
        zipf.write(file, file.name)
        print(f"Added to zip: {file.name}")

print("\n" + "="*60)
print("Downloading zip file...")
print("="*60)
files.download(zip_filename)

print("\n✓ Download complete!")
print("\nNext steps:")
print("1. Extract the zip file")
print("2. Copy the .jpg files to:")
print("   DATA/custom/test/image-densepose/")
print("3. Run inference with PromptDresser!")

In [ ]:
# Create zip file
zip_filename = 'densepose_results.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in Path('output_densepose').glob('*.jpg'):
        zipf.write(file, file.name)
        print(f"Added to zip: {file.name}")

print("\n" + "="*60)
print("Downloading zip file...")
print("="*60)
files.download(zip_filename)

print("\n✓ Download complete!")
print("\nNext steps:")
print("1. Extract the zip file")
print("2. Copy the .jpg files to:")
print("   DATA/custom/test/image-densepose/")
print("3. Run inference with PromptDresser!")

## Step 7: Download Generated DensePose Files

This will create a zip file with all DensePose images and download it.

In [ ]:
# Display original and DensePose side-by-side
output_files = sorted(Path('output_densepose').glob('*.jpg'))

fig, axes = plt.subplots(len(output_files), 2, figsize=(10, 5*len(output_files)))

if len(output_files) == 1:
    axes = axes.reshape(1, -1)

for i, output_file in enumerate(output_files):
    # Original image
    img_name = output_file.stem + '.png'
    orig_path = Path('input_images') / img_name
    orig_img = cv2.imread(str(orig_path))
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    
    # DensePose image
    dense_img = cv2.imread(str(output_file))
    dense_img = cv2.cvtColor(dense_img, cv2.COLOR_BGR2RGB)
    
    # Display
    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title(f'Original: {img_name}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(dense_img)
    axes[i, 1].set_title(f'DensePose: {output_file.name}')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ Showing {len(output_files)} DensePose visualizations")

## Step 6: Preview Generated DensePose Images

In [ ]:
def generate_densepose(image_path, predictor):
    """Generate DensePose visualization for an image"""
    # Read image
    image = cv2.imread(str(image_path))
    if image is None:
        print(f"    ✗ Failed to read: {image_path}")
        return None
    
    # Run prediction
    with torch.no_grad():
        outputs = predictor(image)["instances"]
    
    # Check if person detected
    if len(outputs) == 0:
        print(f"    ⚠️  No person detected in image")
        # Return blank image with same dimensions
        return np.zeros_like(image)
    
    # Extract DensePose results
    extractor = DensePoseResultExtractor()
    results = extractor(outputs)
    
    if len(results) == 0:
        print(f"    ⚠️  No DensePose result extracted")
        return np.zeros_like(image)
    
    # Visualize
    visualizer = DensePoseResultsFineSegmentationVisualizer()
    vis_image = visualizer.visualize(image, results[0])
    
    return vis_image


# Process all uploaded images
input_dir = Path('input_images')
output_dir = Path('output_densepose')

image_files = sorted(input_dir.glob('*.png'))

print("="*60)
print("Generating DensePose visualizations...")
print("="*60 + "\n")

for i, img_path in enumerate(image_files, 1):
    print(f"[{i}/{len(image_files)}] Processing: {img_path.name}")
    
    # Generate DensePose
    vis_image = generate_densepose(img_path, predictor)
    
    if vis_image is not None:
        # Save as .jpg (matching VITON-HD format)
        output_name = img_path.stem + '.jpg'
        output_path = output_dir / output_name
        cv2.imwrite(str(output_path), vis_image)
        print(f"    ✓ Saved: {output_name}")
        print(f"    Size: {vis_image.shape[1]}x{vis_image.shape[0]}\n")
    else:
        print(f"    ✗ Failed to process\n")

print("="*60)
print("✓ DensePose generation complete!")
print("="*60)

## Step 5: Generate DensePose Visualizations

This will process each image and create DensePose maps.

In [ ]:
# Create directories
os.makedirs('input_images', exist_ok=True)
os.makedirs('output_densepose', exist_ok=True)

print("Please upload your 4 custom images:")
print("Expected files: 0000.png, 0009.png, 0012.png, 0016.png")
print("\n" + "="*60)

uploaded = files.upload()

# Move uploaded files to input directory
for filename in uploaded.keys():
    with open(f'input_images/{filename}', 'wb') as f:
        f.write(uploaded[filename])
    print(f"✓ Uploaded: {filename}")

print("="*60)
print(f"Total files uploaded: {len(uploaded)}")

## Step 4: Upload Your Custom Images

Click "Choose Files" and upload your 4 person images:
- 0000.png
- 0009.png
- 0012.png
- 0016.png

In [ ]:
cfg = get_cfg()
add_densepose_config(cfg)

# Use R_50_FPN_s1x model (good balance of speed and accuracy)
cfg.merge_from_file("detectron2/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml")
cfg.MODEL.WEIGHTS = "https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl"
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading DensePose model...")
predictor = DefaultPredictor(cfg)
print("✓ Model loaded and ready!")
print(f"Using device: {cfg.MODEL.DEVICE.upper()}")

## Step 3: Configure DensePose Model

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files
import zipfile
import os

# Detectron2 imports
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

# DensePose imports
sys.path.insert(0, 'detectron2/projects/DensePose')
from densepose import add_densepose_config
from densepose.vis.densepose_results import DensePoseResultsFineSegmentationVisualizer
from densepose.vis.extractor import DensePoseResultExtractor

print("✓ All libraries imported successfully!")

## Step 2: Import Libraries and Setup

In [ ]:
import sys
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install Detectron2
print("\n" + "="*60)
print("Installing Detectron2...")
print("="*60)
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

# Install DensePose
print("\n" + "="*60)
print("Installing DensePose...")
print("="*60)
!git clone https://github.com/facebookresearch/detectron2
!pip install -e detectron2/projects/DensePose

print("\n✓ Installation complete!")

## Step 1: Install Detectron2 and DensePose

This takes about 2-3 minutes. You'll see compilation messages - this is normal.

# DensePose Generator for PromptDresser Custom Images

This notebook generates DensePose visualizations for your custom images using Google Colab's free GPU.

**Instructions:**
1. Run cells in order from top to bottom
2. Upload your 4 custom images (0000.png, 0009.png, 0012.png, 0016.png)
3. Download the generated DensePose files
4. Copy them to your local `DATA/custom/test/image-densepose/` folder

**Runtime:** GPU recommended (Runtime → Change runtime type → GPU)